<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/English_to_hindi_using_encoder_decoder/blob/main/Hindi_to_english.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [130]:
import torch
import torch.nn as nn
import torch.optim as optim
import random

In [87]:
# #!/bin/bash
# !kaggle datasets download preetviradiya/english-hindi-dataset

In [88]:
# !unzip english-hindi-dataset.zip

In [89]:
import pandas as pd

In [90]:
df = pd.read_csv("Dataset_English_Hindi.csv")

In [91]:
df.head()

,English,Hindi
0,Help!,बचाओ!
1,Jump.,उछलो.
2,Jump.,कूदो.
3,Jump.,छलांग.
4,Hello!,नमस्ते।


In [92]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130476 entries, 0 to 130475
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   English  130474 non-null  object
 1   Hindi    130164 non-null  object
dtypes: object(2)
memory usage: 2.0+ MB


In [93]:
df["English"] = df["English"].str.lower().str.split()
df["Hindi"] = df["Hindi"].str.split()

In [94]:
df.head()

,English,Hindi
0,[help!],[बचाओ!]
1,[jump.],[उछलो.]
2,[jump.],[कूदो.]
3,[jump.],[छलांग.]
4,[hello!],[नमस्ते।]


In [95]:
df["Hindi"] = df["Hindi"].apply(
    lambda x: ["<start>"] + x + ["<end>"] if isinstance(x, list) else ["<start>", "<end>"]
    )

In [96]:
df.head()

,English,Hindi
0,[help!],"[<start>, बचाओ!, <end>]"
1,[jump.],"[<start>, उछलो., <end>]"
2,[jump.],"[<start>, कूदो., <end>]"
3,[jump.],"[<start>, छलांग., <end>]"
4,[hello!],"[<start>, नमस्ते।, <end>]"


In [97]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130476 entries, 0 to 130475
Data columns (total 2 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   English  130474 non-null  object
 1   Hindi    130476 non-null  object
dtypes: object(2)
memory usage: 2.0+ MB


In [98]:
df_subset = df.head(5000)

In [99]:
vocab_english = []
for i , sentance in enumerate(df_subset["English"]) :
  vocab_english.extend(sentance)
  # print(sentance)
  # if i == 5000 :break
vocab_english = list(set(vocab_english))
vocab_english = ["<pad>" , "<UNK>"] + vocab_english
print(vocab_english)

['<pad>', '<UNK>', 'of', 'vishwa', 'not),', 'control', 'cave.', '-pace,', 'morning.', '27', "'sum", 'explain.', 'imposed', 'geypadoan', 'pendulum', 'treaty', 'photosynth,', 'everywhere', 'barabaki', 'brother', 'matter', 'caustic', 'london?', 'bath.', 'identities', 'nice', 'knife', 'run.', 'wine.', 'strange', 'me.', 'rome', 'you', 'traitor.', '“”global', 'terrorists', '1979-80', 'represents', 'drona', 'terrorism.', 'discomfort', 'squirrel', 'lachmi', 'inflict', 'combined', 'highways', 'drinks?', 'infrastructure', 'decorated', '5,00,000', 'mean', 'insurgency', 'customs', 'maulana', 'foreign-made', 'loyalty', 'matter?', 'disappointed.', 'natures', 'rolling', 'earlier.', 'bipin', 'singer', 'visvabharati', 'imperialist', 'japanese.', '265', 'industrial', 'salient', 'pardon', '“”chandravanshi-suryavanshi“”', 'resolutions', 'seek', 'abstract.', 'grand', 'memoirs', '1980s.', '[unclear],', 'mandaley', 'markup', 'steam', 'crudest', 'inevitable', 'flow', 'profession', 'eds.)', 'skies', 'represent

In [100]:
vocab_hindi = []
for i , sentance in enumerate(df_subset["Hindi"]) :
  vocab_hindi.extend(sentance)

  # print(sentance)
  if i == 5000 :break
vocab_hindi = list(set(vocab_hindi))
vocab_hindi = ["<pad>" , "<start>" , "<end>"] + vocab_hindi
print(vocab_hindi)
print(len(vocab_hindi))


['<pad>', '<start>', '<end>', 'लेना', 'स्टेडियमों', 'गणेश', 'सेवन,', 'of', 'याचिका', 'भयंदर', 'नाइजीरियाई', 'ट्रेड', 'ऑस्ट्रेलिया', 'conduct)निषिद्ध', 'देकर', '27', 'कुरियना', 'सत्तासीन', 'देखा,', 'शक्तियों', 'सबको', 'समालोचना', 'उठानी', 'पार्टनर', 'भार', 'सोचो', 'नेहरु', 'बतादेना।', 'जनसंख्या', 'आधे', 'क्लब', 'तर्कसंगत', 'चला', 'मिनट', 'साZथ', 'यव', 'महानता', 'सकलेश', 'आकृति', '1979-80', 'प्लान', 'हटा', 'मिलाली', 'जीते', 'गाने', 'बुलाई', 'नन्हाँ', 'संसार', 'मैमोग्राफ', 'cloud)', 'हिलो', 'सिस्टर', 'उस्मान', 'उम्र', 'बल्ख', 'की,', 'हुये', 'मेहराबाकार', 'स्कूल', 'पहुँचा', 'उतरेगा।', 'काउंटर', 'हवाऋ', 'चुना', 'असुरक्षित', 'ब्रोड', 'नियुक्त', 'चाहता।', 'खिलजी', 'कैदी', 'सादा', 'कम्प्यूटर', 'रोना', 'टीम', 'समझदार', 'सपनों', 'आच्छादित', 'है?', 'चलना', 'राजस्थानी', 'सहभागिता', 'जज़िया', '(carcinogen)', 'मंड़पम', 'हें', 'आपसे', 'बनाओ।', 'आज़माती', 'मम्मी-पापा', 'प्रोग्राम', 'डालोगे', 'पसन्द', 'टाइम', 'गतिविधि', 'अथवा', 'मोटरसाईकल', 'ब्राउज़िंग', 'डरते', 'गेंदबाज', 'वैध', 'रौंद', 'भद्दा', 'खतम'

In [101]:
eng_word_2_idx = {word : idx for idx , word in enumerate(vocab_english)}
print(eng_word_2_idx)

{'<pad>': 0, '<UNK>': 1, 'of': 2, 'vishwa': 3, 'not),': 4, 'control': 5, 'cave.': 6, '-pace,': 7, 'morning.': 8, '27': 9, "'sum": 10, 'explain.': 11, 'imposed': 12, 'geypadoan': 13, 'pendulum': 14, 'treaty': 15, 'photosynth,': 16, 'everywhere': 17, 'barabaki': 18, 'brother': 19, 'matter': 20, 'caustic': 21, 'london?': 22, 'bath.': 23, 'identities': 24, 'nice': 25, 'knife': 26, 'run.': 27, 'wine.': 28, 'strange': 29, 'me.': 30, 'rome': 31, 'you': 32, 'traitor.': 33, '“”global': 34, 'terrorists': 35, '1979-80': 36, 'represents': 37, 'drona': 38, 'terrorism.': 39, 'discomfort': 40, 'squirrel': 41, 'lachmi': 42, 'inflict': 43, 'combined': 44, 'highways': 45, 'drinks?': 46, 'infrastructure': 47, 'decorated': 48, '5,00,000': 49, 'mean': 50, 'insurgency': 51, 'customs': 52, 'maulana': 53, 'foreign-made': 54, 'loyalty': 55, 'matter?': 56, 'disappointed.': 57, 'natures': 58, 'rolling': 59, 'earlier.': 60, 'bipin': 61, 'singer': 62, 'visvabharati': 63, 'imperialist': 64, 'japanese.': 65, '265': 

In [102]:
hindi_word_2_idx = {word : idx for idx , word in enumerate(vocab_hindi)}
print(hindi_word_2_idx)

{'<pad>': 0, '<start>': 683, '<end>': 5710, 'लेना': 3, 'स्टेडियमों': 4, 'गणेश': 5, 'सेवन,': 6, 'of': 7, 'याचिका': 8, 'भयंदर': 9, 'नाइजीरियाई': 10, 'ट्रेड': 11, 'ऑस्ट्रेलिया': 12, 'conduct)निषिद्ध': 13, 'देकर': 14, '27': 15, 'कुरियना': 16, 'सत्तासीन': 17, 'देखा,': 18, 'शक्तियों': 19, 'सबको': 20, 'समालोचना': 21, 'उठानी': 22, 'पार्टनर': 23, 'भार': 24, 'सोचो': 25, 'नेहरु': 26, 'बतादेना।': 27, 'जनसंख्या': 28, 'आधे': 29, 'क्लब': 30, 'तर्कसंगत': 31, 'चला': 32, 'मिनट': 33, 'साZथ': 34, 'यव': 35, 'महानता': 36, 'सकलेश': 37, 'आकृति': 38, '1979-80': 39, 'प्लान': 40, 'हटा': 41, 'मिलाली': 42, 'जीते': 43, 'गाने': 44, 'बुलाई': 45, 'नन्हाँ': 46, 'संसार': 47, 'मैमोग्राफ': 48, 'cloud)': 49, 'हिलो': 50, 'सिस्टर': 51, 'उस्मान': 52, 'उम्र': 53, 'बल्ख': 54, 'की,': 55, 'हुये': 56, 'मेहराबाकार': 57, 'स्कूल': 58, 'पहुँचा': 59, 'उतरेगा।': 60, 'काउंटर': 61, 'हवाऋ': 62, 'चुना': 63, 'असुरक्षित': 64, 'ब्रोड': 65, 'नियुक्त': 66, 'चाहता।': 67, 'खिलजी': 68, 'कैदी': 69, 'सादा': 70, 'कम्प्यूटर': 71, 'रोना': 72, 'टीम': 73,

In [103]:
def word_to_number(sentance , vocab):
  value = []
  for i in sentance:
    value.append(vocab[i])
  return value

In [104]:
word_to_number(["reminded" , "and" , "in"] , eng_word_2_idx)

[7990, 9546, 819]

In [105]:
# ["English"] , df_subset["hindi"]

In [106]:
temp_for_eng = []
temp_for_hindi = []
for i , j in zip(df_subset["English"] , df_subset["Hindi"]):
  temp_for_eng.append(word_to_number(i , eng_word_2_idx))
  temp_for_hindi.append(word_to_number(j , hindi_word_2_idx))


# print(temp_for_eng)
df_subset["English"] = temp_for_eng
df_subset["Hindi"] = temp_for_hindi

/tmp/ipykernel_1805/3841854393.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset["English"] = temp_for_eng
/tmp/ipykernel_1805/3841854393.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_subset["Hindi"] = temp_for_hindi


In [107]:
df_subset["English"][100]
df_subset["Hindi"][100]

[683, 9800, 1355, 10181, 1183, 5548, 5710]

In [108]:
df_subset.head()

,English,Hindi
0,[2927],"[683, 2071, 5710]"
1,[7586],"[683, 2706, 5710]"
2,[7586],"[683, 9364, 5710]"
3,[7586],"[683, 9728, 5710]"
4,[1949],"[683, 6902, 5710]"


In [109]:
# from torch.nn.utils.rnn import pad_sequence

# padded = pad_sequence(
#     sequences,
#     batch_first=True,
#     padding_value = eng_word_2_idx["<PAD>"]
# )

In [110]:
from torch.utils.data import DataLoader , Dataset

In [111]:
class dataset(Dataset):
  def __init__(self , df):
    self.x = df["English"]
    self.y = df["Hindi"]

    self.x = torch.tensor(self.x , dtype=torch.float)
    self.y = torch.tensor(self.y , dtype=torch.float)

  def __len__(self):
    return len(self.x)
  def __getitem__(self , index):

    return self.x[index] , self.y[index]

In [112]:
from torch.utils.data import DataLoader , Dataset
import torch

class dataset(Dataset):
  def __init__(self , df):
    self.x = df["English"].tolist()
    self.y = df["Hindi"].tolist()

  def __len__(self):
    return len(self.x)

  def __getitem__(self , index):
    return torch.tensor(self.x[index] , dtype=torch.long) , torch.tensor(self.y[index] , dtype=torch.long)

In [113]:
from torch.nn.utils.rnn import pad_sequence
import torch

PAD_IDX = 0

def collate_fn(batch):

    src = [torch.tensor(x[0]) for x in batch]
    trg = [torch.tensor(x[1]) for x in batch]

    src = pad_sequence(
        src,
        batch_first=True,
        padding_value=PAD_IDX
    )

    trg = pad_sequence(
        trg,
        batch_first=True,
        padding_value=PAD_IDX
    )

    return src, trg

In [114]:
train_dataset = dataset(df_subset)
dataLoader = DataLoader(train_dataset , batch_size=32 , shuffle=True , collate_fn=collate_fn)

In [115]:
for i , (m, n) in enumerate(dataLoader):
  print(m.size())
  if i == 5 : break

torch.Size([32, 29])
torch.Size([32, 58])
torch.Size([32, 27])
torch.Size([32, 60])
torch.Size([32, 47])
torch.Size([32, 37])


/tmp/ipykernel_1805/3754108929.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  src = [torch.tensor(x[0]) for x in batch]
/tmp/ipykernel_1805/3754108929.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  trg = [torch.tensor(x[1]) for x in batch]


In [116]:
class Encoder(nn.Module):
  def __init__(self , vocab_size):
    super().__init__()
    self.emb = nn.Embedding(vocab_size , 500)
    self.GRU = nn.GRU(500 , 200 , 3 , batch_first=True)

  def forward(self , x):
    emb = self.emb(x)
    output , hidden = self.GRU(emb)
    return hidden

In [117]:
class Decoder(nn.Module):
  def __init__(self , vocab_size):
    super().__init__()
    self.emb = nn.Embedding(vocab_size , 500)
    self.gru = nn.GRU(500 , 200 , 2 , batch_first=True)
    self.fc = nn.Linear(200 , vocab_size)

  def forward(self , prev_prediction , hidden):
    emb = self.emb(prev_prediction)
    output , hidden = self.gru(emb , hidden)
    prediction = self.fc(output)
    return prediction , hidden

In [118]:
len(vocab_english)

10377

In [136]:
class Seq2Seq(nn.Module):
  def __init__(self , vocab_english , vocab_hindi):
    super().__init__()
    self.encoder = Encoder(len(vocab_english))
    self.decoder = Decoder(len(vocab_hindi))

  def forward(self , english_src , hindi_target):
    hidden = self.encoder(english_src)
    decoder_input = hindi_target[: , 0]

    predictions = []

    target_len = hindi_target.shape[1]

    for t in range(1, target_len):

      prediction , hidden = self.decoder(decoder_input , hidden)
      predictions.append(prediction)

      teacher_forcing_ratio = 0.7
      teacher_force = random.random() < teacher_forcing_ratio

      if teacher_force:
        decoder_input = hindi_target[: , t]
      else:
        decoder_input = prediction.argmax(dim=1)

    predictions = torch.stack(predictions, dim=1)
    return predictions